# Solar Flares: MHD Processes — Implementation / 태양 플레어 MHD 과정 구현

**Paper**: Shibata, K. & Magara, T., *Living Reviews in Solar Physics* **8**, 6 (2011).

**English.** This notebook reproduces the core quantitative results of Shibata & Magara (2011): the Sweet-Parker and Petschek reconnection-rate scalings, the tearing-mode growth rate, the plasmoid-instability criterion (Sc ~ 10^4), a CSHKP geometry cartoon, and a 1D Harris current-sheet equilibrium.

**한국어.** 이 노트북은 Shibata & Magara (2011)의 핵심 정량 결과를 재현한다: Sweet-Parker 및 Petschek 재결합 속도 스케일링, tearing 모드 성장률, 플라즈모이드 불안정성 임계값 (S_c ~ 10^4), CSHKP 기하 cartoon, 1D Harris current sheet 평형.

In [ ]:
"""Imports and plotting setup.

This cell sets up numpy for numerical computation and matplotlib for
visualization. All constants and subsequent calculations follow Gaussian-CGS
units as used in the Shibata & Magara (2011) review.
"""
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, Ellipse, Rectangle

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

## 1. Sweet-Parker vs Petschek reconnection rate / Sweet-Parker 대 Petschek 재결합 속도

**English.** We plot the dimensionless reconnection rate $M_A = v_i/v_A$ as a function of Lundquist number $R_m = v_A L / \eta$ for both the Sweet-Parker scaling $M_A \simeq R_m^{-1/2}$ (Eq. 19 of the paper) and the Petschek scaling $M_A \simeq \pi/(8 \ln R_m)$ (Eq. 21). At the typical coronal value $R_m \sim 10^{14}$ the two scalings differ by six orders of magnitude — the reason why pure Sweet-Parker cannot explain flare timescales.

**한국어.** 무차원 재결합 속도 $M_A = v_i/v_A$를 Lundquist 수 $R_m = v_A L / \eta$의 함수로 Sweet-Parker 스케일링 $M_A \simeq R_m^{-1/2}$ (논문 Eq. 19)와 Petschek 스케일링 $M_A \simeq \pi/(8 \ln R_m)$ (Eq. 21) 모두에 대해 그린다. 전형적 코로나 값 $R_m \sim 10^{14}$에서 두 스케일링은 6차수 차이가 난다 — 순수 Sweet-Parker가 플레어 시간 규모를 설명하지 못하는 이유이다.

In [ ]:
def sweet_parker_rate(rm):
    """Sweet-Parker dimensionless reconnection rate.

    Args:
        rm: Lundquist number (dimensionless).

    Returns:
        Inflow Alfven Mach number M_A = v_i / v_A = R_m^{-1/2}.
    """
    return rm ** (-0.5)


def petschek_rate(rm):
    """Petschek dimensionless reconnection rate.

    Args:
        rm: Lundquist number (dimensionless).

    Returns:
        Inflow Alfven Mach number M_A = pi / (8 ln R_m).
    """
    return np.pi / (8.0 * np.log(rm))


rm_grid = np.logspace(2, 16, 400)
ma_sp = sweet_parker_rate(rm_grid)
ma_pk = petschek_rate(rm_grid)

fig, ax = plt.subplots(figsize=(7.5, 5))
ax.loglog(rm_grid, ma_sp, label=r"Sweet-Parker: $M_A = R_m^{-1/2}$", lw=2)
ax.loglog(rm_grid, ma_pk, label=r"Petschek: $M_A = \pi / (8\ln R_m)$", lw=2)
ax.axvline(1e14, color="gray", ls=":", label=r"Coronal $R_m \sim 10^{14}$")
ax.axhspan(0.01, 0.1, color="orange", alpha=0.2,
           label="Observed flare rate (0.01 - 0.1)")
ax.set_xlabel("Lundquist number $R_m = v_A L / \\eta$")
ax.set_ylabel("Reconnection rate $M_A = v_i / v_A$")
ax.set_title("Sweet-Parker vs Petschek reconnection rate\nShibata & Magara (2011), Eqs. 19, 21")
ax.legend(loc="best")
plt.tight_layout()
plt.show()

print(f"At coronal R_m = 1e14:")
print(f"  Sweet-Parker M_A = {sweet_parker_rate(1e14):.2e}")
print(f"  Petschek    M_A = {petschek_rate(1e14):.2e}")
print(f"  Ratio           = {petschek_rate(1e14)/sweet_parker_rate(1e14):.2e}")

## 2. Reconnection time comparison / 재결합 시간 비교

**English.** Convert the reconnection rate to a physical reconnection time $t_\text{rec} = t_A / M_A$, where $t_A = L/v_A$ is the Alfven transit time across the current sheet of length $L$. At coronal conditions ($v_A = 2000$ km/s, $L = 10^4$ km, so $t_A \simeq 5$ s), Sweet-Parker gives a timescale comparable to the solar cycle while Petschek matches the observed impulsive-phase duration of minutes.

**한국어.** 재결합 속도를 물리적 재결합 시간 $t_\text{rec} = t_A / M_A$로 변환한다. 여기서 $t_A = L/v_A$는 길이 $L$인 current sheet 가로지르는 Alfven 이동 시간이다. 코로나 조건 ($v_A = 2000$ km/s, $L = 10^4$ km, 따라서 $t_A \simeq 5$ s)에서 Sweet-Parker는 태양 주기와 비슷한 시간 규모를, Petschek는 관측되는 수 분 규모의 임펄시브 단계 지속 시간을 준다.

In [ ]:
def reconnection_time(ma, length_cm, v_alfven_cm_s):
    """Physical reconnection time.

    Args:
        ma: Dimensionless Alfven-Mach reconnection rate.
        length_cm: Current-sheet length in centimeters.
        v_alfven_cm_s: Alfven velocity in cm/s.

    Returns:
        Reconnection time in seconds, t_rec = L / (M_A * v_A).
    """
    return length_cm / (ma * v_alfven_cm_s)


# Coronal parameters from Shibata & Magara Eq. 22
B_gauss = 100.0  # magnetic field strength [G]
n_cm3 = 1.0e10  # number density [cm^-3]
L_cm = 1.0e9   # sheet length [cm] = 10^4 km
eta_cgs = 0.3  # Spitzer diffusivity at T=10^7 K [cm^2/s]

m_proton = 1.673e-24  # g
rho = n_cm3 * m_proton
v_A = B_gauss / np.sqrt(4 * np.pi * rho)  # cm/s
R_m = v_A * L_cm / eta_cgs
t_A = L_cm / v_A

M_sp = sweet_parker_rate(R_m)
M_pk = petschek_rate(R_m)
t_sp = reconnection_time(M_sp, L_cm, v_A)
t_pk = reconnection_time(M_pk, L_cm, v_A)

print("Coronal parameters:")
print(f"  B = {B_gauss} G, n = {n_cm3:.1e} cm^-3, L = {L_cm:.1e} cm")
print(f"  v_A = {v_A/1e5:.0f} km/s")
print(f"  t_A = L/v_A = {t_A:.2f} s")
print(f"  R_m = {R_m:.2e}")
print()
print("Reconnection timescales:")
print(f"  Sweet-Parker: M_A = {M_sp:.2e}, t_rec = {t_sp:.2e} s ({t_sp/3.15e7:.2f} yr)")
print(f"  Petschek:     M_A = {M_pk:.2e}, t_rec = {t_pk:.2e} s ({t_pk/60:.1f} min)")

## 3. Tearing-mode growth rate / Tearing 모드 성장률

**English.** The Furth-Killeen-Rosenbluth (1963) tearing instability of a Harris current sheet grows at a rate intermediate between the Alfven time $\tau_A$ and resistive time $\tau_R$: $\gamma \sim \tau_A^{-2/5} \tau_R^{-3/5}$. In units of $1/\tau_A$ this is $\gamma \tau_A \sim R_m^{-3/5}$. We compare against the ideal rate ($\gamma \tau_A \sim 1$) and pure resistive diffusion rate ($\gamma \tau_A \sim R_m^{-1}$).

**한국어.** Harris current sheet의 Furth-Killeen-Rosenbluth (1963) tearing 불안정성은 Alfven 시간 $\tau_A$와 저항 시간 $\tau_R$ 사이의 중간 속도로 성장한다: $\gamma \sim \tau_A^{-2/5} \tau_R^{-3/5}$. $1/\tau_A$ 단위로 이는 $\gamma \tau_A \sim R_m^{-3/5}$이다. 이상 속도 ($\gamma \tau_A \sim 1$) 및 순수 저항 확산 속도 ($\gamma \tau_A \sim R_m^{-1}$)와 비교한다.

In [ ]:
def tearing_growth_rate(rm):
    """FKR (1963) tearing-mode growth rate in units of 1/tau_A.

    Args:
        rm: Lundquist number.

    Returns:
        Dimensionless growth rate gamma * tau_A ~ R_m^{-3/5}.
    """
    return rm ** (-0.6)


rm_grid = np.logspace(2, 14, 300)
gamma_tearing = tearing_growth_rate(rm_grid)
gamma_resistive = 1.0 / rm_grid  # pure resistive
gamma_ideal = np.ones_like(rm_grid)

fig, ax = plt.subplots(figsize=(7.5, 5))
ax.loglog(rm_grid, gamma_ideal, label=r"Ideal MHD: $\gamma\tau_A \sim 1$",
          ls="--", color="gray")
ax.loglog(rm_grid, gamma_tearing,
          label=r"Tearing (FKR 1963): $\gamma\tau_A \sim R_m^{-3/5}$", lw=2)
ax.loglog(rm_grid, gamma_resistive,
          label=r"Resistive diffusion: $\gamma\tau_A \sim R_m^{-1}$",
          ls=":", color="red")
ax.axvline(1e4, color="purple", ls="--",
           label=r"Plasmoid-instability threshold $S_c \sim 10^4$")
ax.set_xlabel("Lundquist number $R_m$")
ax.set_ylabel(r"Growth rate $\gamma \tau_A$")
ax.set_title("Instability growth rates vs Lundquist number")
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()

## 4. Plasmoid instability threshold / 플라즈모이드 불안정성 임계값

**English.** A Sweet-Parker current sheet becomes violently unstable to secondary tearing once $S > S_c \simeq 10^4$ (Loureiro et al. 2007, Bhattacharjee et al. 2009, Shibata & Tanuma 2001). The number of plasmoids scales as $N \sim S^{3/8}$, and the effective reconnection rate saturates at $M_\text{eff} \sim 10^{-2}$ — independent of $S$. This resolves the Sweet-Parker-Petschek tension because no steady long Sweet-Parker sheet can exist at coronal $S = 10^{14}$.

**한국어.** Sweet-Parker current sheet는 $S > S_c \simeq 10^4$ (Loureiro et al. 2007, Bhattacharjee et al. 2009, Shibata & Tanuma 2001)이 되면 2차 tearing에 대해 격렬하게 불안정해진다. 플라즈모이드 수는 $N \sim S^{3/8}$로 스케일되고, 유효 재결합 속도는 $M_\text{eff} \sim 10^{-2}$로 — $S$에 무관하게 — 포화한다. 이는 코로나 $S = 10^{14}$에서 정상 긴 Sweet-Parker sheet가 존재할 수 없기에 Sweet-Parker-Petschek 긴장을 해결한다.

In [ ]:
def effective_reconnection_rate(s, s_crit=1e4, m_plasmoid=1e-2):
    """Effective reconnection rate in the plasmoid-dominated regime.

    Below S_c the sheet is stable and follows Sweet-Parker scaling. Above
    S_c the sheet fragments into plasmoids and the rate saturates at a
    Petschek-like value near 10^-2 (Bhattacharjee et al. 2009).

    Args:
        s: Lundquist number.
        s_crit: Critical Lundquist number for plasmoid instability.
        m_plasmoid: Saturated reconnection rate above S_c.

    Returns:
        Effective dimensionless reconnection rate.
    """
    sp_rate = s ** (-0.5)
    return np.where(s < s_crit, sp_rate, np.full_like(s, m_plasmoid))


def number_of_plasmoids(s, s_crit=1e4):
    """Number of plasmoids in a fragmented current sheet, N ~ S^{3/8}.

    Args:
        s: Lundquist number.
        s_crit: Critical Lundquist number for plasmoid instability.

    Returns:
        Integer number of plasmoids; below S_c returns 1 (single sheet).
    """
    n = np.where(s < s_crit, 1.0, (s / s_crit) ** 0.375)
    return n


s_grid = np.logspace(2, 16, 400)
m_eff = effective_reconnection_rate(s_grid)
n_plasm = number_of_plasmoids(s_grid)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].loglog(s_grid, s_grid ** (-0.5), ls=":", color="gray",
               label="Sweet-Parker (unstable for S > S_c)")
axes[0].loglog(s_grid, m_eff, lw=2, color="crimson",
               label="Effective (plasmoid-mediated)")
axes[0].axvline(1e4, ls="--", color="purple", label=r"$S_c \sim 10^4$")
axes[0].axvline(1e14, ls=":", color="black", label=r"Coronal $S \sim 10^{14}$")
axes[0].set_xlabel("Lundquist number $S$")
axes[0].set_ylabel("Effective reconnection rate $M_\\text{eff}$")
axes[0].set_title("Plasmoid-mediated reconnection rate")
axes[0].legend()

axes[1].loglog(s_grid, n_plasm, lw=2, color="teal")
axes[1].axvline(1e4, ls="--", color="purple", label=r"$S_c \sim 10^4$")
axes[1].set_xlabel("Lundquist number $S$")
axes[1].set_ylabel("Number of plasmoids $N \\sim S^{3/8}$")
axes[1].set_title("Plasmoid count in fragmented sheet")
axes[1].legend()
plt.tight_layout()
plt.show()

print(f"At coronal S = 1e14, expected plasmoid count: N ~ {number_of_plasmoids(np.array([1e14]))[0]:.0f}")

## 5. CSHKP flare geometry cartoon / CSHKP 플레어 기하 cartoon

**English.** Reproduce the Shibata et al. (1995) modified CSHKP diagram (Figure 3b of Shibata & Magara 2011): rising plasmoid at top, reconnection X-point with current sheet below it, bidirectional reconnection jets, fast shock (HXR loop-top source) above the SXR loop, and H-alpha footpoint ribbons below. This single diagram summarizes the entire flare energy chain.

**한국어.** Shibata et al. (1995)의 수정 CSHKP 다이어그램을 재현한다 (Shibata & Magara 2011의 Figure 3b): 상단의 상승 플라즈모이드, 그 아래 current sheet가 있는 재결합 X점, 양방향 reconnection jet, SXR 루프 위의 빠른 충격파 (HXR 루프톱 source), 그 아래의 H-alpha 발자국 ribbons. 이 단일 다이어그램이 전체 플레어 에너지 사슬을 요약한다.

In [ ]:
def draw_cshkp_diagram():
    """Draw the modified CSHKP flare geometry as a cartoon.

    Reproduces the schematic of Shibata et al. (1995) / Figure 3b of
    Shibata & Magara (2011): plasmoid, current sheet with bidirectional
    reconnection jets, SXR loop, fast shock, HXR loop-top source, and
    chromospheric footpoint ribbons.
    """
    fig, ax = plt.subplots(figsize=(8, 10))
    ax.set_xlim(-5, 5)
    ax.set_ylim(-0.5, 12)
    ax.set_aspect("equal")

    # Chromosphere (ground)
    ax.axhspan(-0.5, 0, color="tan", alpha=0.5)
    ax.text(0, -0.3, "Chromosphere", ha="center", fontsize=10)

    # H-alpha two ribbons
    ax.add_patch(Rectangle((-2.5, 0), 1.0, 0.1, color="red"))
    ax.add_patch(Rectangle((1.5, 0), 1.0, 0.1, color="red"))
    ax.text(-2.0, -0.2, r"H$\alpha$ ribbon", ha="center", fontsize=9, color="red")
    ax.text(2.0, -0.2, r"H$\alpha$ ribbon", ha="center", fontsize=9, color="red")

    # SXR loop (arcade)
    theta = np.linspace(0, np.pi, 100)
    loop_r = 2.5
    loop_cx, loop_cy = 0, 0
    lx = loop_cx + loop_r * np.cos(theta)
    ly = loop_cy + loop_r * np.sin(theta)
    ax.plot(lx, ly, color="orange", lw=4, label="SXR loop")

    # Fast shock / HXR loop-top source
    ax.add_patch(Ellipse((0, 3.2), 1.2, 0.4, color="magenta", alpha=0.6))
    ax.text(2.4, 3.2, "HXR loop-top\nsource (fast shock)",
            fontsize=9, color="magenta", va="center")

    # Current sheet
    ax.plot([0, 0], [3.6, 7.8], color="blue", lw=2, ls="--")
    ax.text(0.3, 5.5, "Current sheet\n(reconnection site)",
            fontsize=9, color="blue")

    # Reconnection X point
    ax.plot(0, 5.7, "x", color="black", markersize=14, markeredgewidth=2)
    ax.text(-2.3, 5.7, "Reconnection\nX-point", fontsize=9, ha="right")

    # Reconnection jets
    ax.annotate("", xy=(0, 3.8), xytext=(0, 5.4),
                arrowprops=dict(arrowstyle="->", color="green", lw=2.5))
    ax.annotate("", xy=(0, 7.6), xytext=(0, 6.0),
                arrowprops=dict(arrowstyle="->", color="green", lw=2.5))
    ax.text(-1.2, 4.5, r"$v_\mathrm{jet}\sim v_A$", color="green", fontsize=10)
    ax.text(-1.2, 7.0, r"$v_\mathrm{jet}\sim v_A$", color="green", fontsize=10)

    # Plasmoid
    ax.add_patch(Ellipse((0, 9.0), 2.0, 1.0,
                         edgecolor="purple", facecolor="lavender", lw=2))
    ax.text(2.2, 9.0, "Plasmoid / filament", fontsize=10, color="purple",
            va="center")
    ax.annotate("", xy=(0, 10.5), xytext=(0, 9.7),
                arrowprops=dict(arrowstyle="->", color="purple", lw=2))
    ax.text(0.3, 10.2, r"$v_\mathrm{plasmoid}$", color="purple", fontsize=10)

    # Inflow arrows
    for sign in (-1, 1):
        ax.annotate("", xy=(sign * 0.3, 5.7), xytext=(sign * 2.5, 5.7),
                    arrowprops=dict(arrowstyle="->", color="steelblue", lw=2))
    ax.text(-3.3, 5.9, r"Inflow $v_i$", color="steelblue", fontsize=10)

    # Conduction front / evaporation
    ax.annotate("", xy=(-1.8, 0.2), xytext=(-1.8, 2.2),
                arrowprops=dict(arrowstyle="->", color="brown", lw=1.5))
    ax.text(-3.7, 1.2, "Thermal\nconduction", color="brown", fontsize=9)
    ax.annotate("", xy=(-1.4, 2.2), xytext=(-1.4, 0.2),
                arrowprops=dict(arrowstyle="->", color="darkred", lw=1.5))
    ax.text(-0.9, 1.2, "Evaporation\nupflow", color="darkred", fontsize=9)

    ax.set_title("Modified CSHKP flare geometry\n(Shibata et al. 1995; S&M 2011, Fig. 3b)",
                 fontsize=12)
    ax.set_xlabel("Horizontal distance (arb.)")
    ax.set_ylabel("Height (arb.)")
    ax.grid(False)
    plt.tight_layout()
    plt.show()


draw_cshkp_diagram()

## 6. Harris current-sheet equilibrium / Harris current sheet 평형

**English.** The canonical 1D equilibrium for a current sheet is the Harris sheet: $B_x(z) = B_0 \tanh(z/a)$ reversing across a scale $a$, with associated current density $j_y \propto \mathrm{sech}^2(z/a)$ and particle density $n(z) = n_0 \mathrm{sech}^2(z/a) + n_\infty$ chosen to keep total pressure (magnetic + gas) constant. This is the equilibrium that tearing and plasmoid instabilities perturb.

**한국어.** Current sheet의 정준 1D 평형은 Harris sheet이다: $B_x(z) = B_0 \tanh(z/a)$가 규모 $a$에 걸쳐 반전되고, 이에 동반되는 전류 밀도 $j_y \propto \mathrm{sech}^2(z/a)$와 입자 밀도 $n(z) = n_0 \mathrm{sech}^2(z/a) + n_\infty$는 총 압력 (자기 + 기체)을 일정하게 유지하도록 선택된다. 이것이 tearing과 플라즈모이드 불안정성이 교란하는 평형이다.

In [ ]:
def harris_sheet(z, a=1.0, b0=1.0, n0=1.0, n_inf=0.2, T=1.0):
    """Harris current-sheet equilibrium in 1D.

    Args:
        z: Array of vertical positions (in units of sheet half-thickness).
        a: Sheet half-thickness (default 1.0, normalized).
        b0: Asymptotic magnetic field amplitude.
        n0: Central excess density.
        n_inf: Background (asymptotic) density.
        T: Temperature (kept constant, isothermal).

    Returns:
        Dictionary with Bx, jy, n, p_mag, p_gas, p_total fields.
    """
    bx = b0 * np.tanh(z / a)
    jy = (b0 / a) * (1.0 / np.cosh(z / a)) ** 2  # arbitrary units (c/4pi absorbed)
    n = n0 * (1.0 / np.cosh(z / a)) ** 2 + n_inf
    p_mag = bx ** 2 / 2.0  # in units where 8 pi is absorbed
    p_gas = n * T
    p_total = p_mag + p_gas
    return dict(Bx=bx, jy=jy, n=n, p_mag=p_mag, p_gas=p_gas, p_total=p_total)


z = np.linspace(-4, 4, 400)
sol = harris_sheet(z, a=1.0, b0=1.0, n0=1.0, n_inf=0.2, T=0.5)

fig, axes = plt.subplots(2, 2, figsize=(11, 7.5), sharex=True)
axes[0, 0].plot(z, sol["Bx"], lw=2, color="navy")
axes[0, 0].set_ylabel(r"$B_x / B_0$")
axes[0, 0].set_title(r"Magnetic field: $B_x = B_0 \tanh(z/a)$")
axes[0, 0].axhline(0, color="gray", lw=0.5)

axes[0, 1].plot(z, sol["jy"], lw=2, color="crimson")
axes[0, 1].set_ylabel(r"$j_y$ (arb.)")
axes[0, 1].set_title(r"Current density: $j_y \propto \mathrm{sech}^2(z/a)$")

axes[1, 0].plot(z, sol["n"], lw=2, color="green")
axes[1, 0].set_xlabel(r"$z/a$")
axes[1, 0].set_ylabel(r"$n / n_0$")
axes[1, 0].set_title(r"Density: $n = n_0\,\mathrm{sech}^2(z/a) + n_\infty$")

axes[1, 1].plot(z, sol["p_mag"], lw=2, color="navy", label="Magnetic")
axes[1, 1].plot(z, sol["p_gas"], lw=2, color="green", label="Gas")
axes[1, 1].plot(z, sol["p_total"], lw=2, color="black", ls="--",
                label="Total (should be constant)")
axes[1, 1].set_xlabel(r"$z/a$")
axes[1, 1].set_ylabel("Pressure (arb.)")
axes[1, 1].set_title("Pressure balance across sheet")
axes[1, 1].legend(fontsize=9)

fig.suptitle("1D Harris current-sheet equilibrium", fontsize=13)
plt.tight_layout()
plt.show()

# Verify pressure balance (with T tuned for exact balance needs n0 T = B0^2/2)
print(f"Pressure at z=0:     p_total = {sol['p_total'][200]:.4f}")
print(f"Pressure at z=+4a:   p_total = {sol['p_total'][-1]:.4f}")
print("(With chosen parameters, not exactly constant — adjust T to enforce balance.)")

## 7. Reconnection energy-release rate / 재결합 에너지 해방률

**English.** The Poynting flux into a diffusion region of size $L$ with inflow speed $v_i$ and field $B_i$ gives the reconnection energy-release rate (Eq. 23 of S&M 2011):
$$\frac{dE}{dt} = \frac{L^2 B_i^2 v_i}{4\pi}.$$
We evaluate this for a grid of flare parameters to reproduce the $\sim 4 \times 10^{28}$ erg/s value observed in impulsive-phase flares (Masuda et al. 1994).

**한국어.** 크기 $L$인 확산 영역으로 유입 속도 $v_i$와 자기장 $B_i$로 들어가는 포인팅 플럭스는 재결합 에너지 해방률을 준다 (S&M 2011 Eq. 23): $dE/dt = L^2 B_i^2 v_i / (4\pi)$. 플레어 매개변수 격자에 대해 평가하여 임펄시브 단계 플레어에서 관측되는 $\sim 4 \times 10^{28}$ erg/s 값을 재현한다 (Masuda et al. 1994).

In [ ]:
def reconnection_power(L_cm, B_gauss, v_i_cm_s):
    """Reconnection Poynting-flux energy release rate.

    Args:
        L_cm: Diffusion region linear size [cm].
        B_gauss: Inflow magnetic field [G].
        v_i_cm_s: Inflow speed [cm/s].

    Returns:
        Energy release rate in erg/s.
    """
    return L_cm ** 2 * B_gauss ** 2 * v_i_cm_s / (4.0 * np.pi)


# Reproduce Shibata & Magara Eq. 23 example
L = 2e9  # cm
B = 100.0  # G
v_i = 100e5  # 100 km/s in cm/s
dE_dt = reconnection_power(L, B, v_i)
print(f"L = {L:.1e} cm, B = {B} G, v_i = {v_i/1e5} km/s")
print(f"dE/dt = {dE_dt:.2e} erg/s")
print(f"(Compare with Shibata & Magara Eq. 23: ~ 4e28 erg/s)")

# Parametric scan: vary L and B at fixed Petschek v_i = 100 km/s
L_grid = np.logspace(8, 10, 50)  # cm
B_grid = np.array([30, 100, 300, 1000])

fig, ax = plt.subplots(figsize=(7.5, 5))
for b in B_grid:
    power = reconnection_power(L_grid, b, v_i)
    ax.loglog(L_grid / 1e5, power, lw=2, label=f"$B = {b}$ G")
ax.axhspan(1e28, 1e29, color="orange", alpha=0.15,
           label="Observed impulsive flare (Masuda 1994)")
ax.set_xlabel("Current-sheet size $L$ [km]")
ax.set_ylabel("$dE/dt$ [erg/s]")
ax.set_title("Reconnection power vs sheet size and field strength\n(Petschek $v_i = 100$ km/s)")
ax.legend()
plt.tight_layout()
plt.show()

## 8. Summary / 요약

**English.** This notebook reproduced the central quantitative results of Shibata & Magara (2011):

1. The Sweet-Parker vs Petschek reconnection rates differ by $\sim 10^6$ at coronal $R_m$.
2. At $R_m = 10^{14}$, Sweet-Parker $t_\text{rec} \sim$ yr while Petschek $t_\text{rec} \sim$ min.
3. Tearing growth $\gamma \tau_A \sim R_m^{-3/5}$ is intermediate between ideal and resistive rates.
4. Above $S_c \sim 10^4$ the plasmoid instability saturates the effective rate at $\sim 10^{-2}$ — a Petschek-like value — independent of $\eta$, resolving the 40-year reconnection puzzle.
5. The CSHKP cartoon captures plasmoid + current sheet + jets + SXR loop + HXR loop-top + H$\alpha$ ribbons.
6. The Harris sheet provides the 1D equilibrium against which reconnection instabilities are evaluated.
7. The reconnection Poynting flux matches observed impulsive-phase flare luminosities.

**한국어.** 이 노트북은 Shibata & Magara (2011)의 핵심 정량 결과를 재현했다:

1. Sweet-Parker와 Petschek 재결합 속도는 코로나 $R_m$에서 $\sim 10^6$ 차이.
2. $R_m = 10^{14}$에서 Sweet-Parker $t_\text{rec} \sim$ 년, Petschek $t_\text{rec} \sim$ 분.
3. Tearing 성장 $\gamma \tau_A \sim R_m^{-3/5}$는 이상과 저항 속도의 중간.
4. $S_c \sim 10^4$ 이상에서 플라즈모이드 불안정성이 유효 속도를 $\sim 10^{-2}$ — Petschek 유사 값 — 로 $\eta$에 무관하게 포화시켜 40년된 재결합 퍼즐을 해결.
5. CSHKP cartoon은 플라즈모이드 + current sheet + jet + SXR 루프 + HXR 루프톱 + H$\alpha$ ribbons를 포착.
6. Harris sheet는 재결합 불안정성을 평가하는 1D 평형을 제공.
7. 재결합 포인팅 플럭스는 관측된 임펄시브 단계 플레어 광도와 일치.